# Классификация новостей по девяти темам

Сравнение словесных и символьных TF-IDF-признаков, LogisticRegression и LinearSVC. Лучший сохранённый результат на валидации: accuracy 0,9502.

Учебная работа НИУ ВШЭ или Яндекс Практикума; исходные формулировки заданий принадлежат авторам курса.

**Воспроизводимость:** Нужен train_data.csv со столбцами title, text, class_id. Метрика относится к валидации, не к внешнему тесту. Формирование submission адаптировано для таблиц с title/text, text или одним текстовым столбцом; новый прогон обучения не выполнялся.

Сохранённые результаты получены до публикации; полное обучение при подготовке репозитория не повторялось.


это **2 ноутбук**, в него входит:
- токенизация, обработка данных, векторизация
- обучение модели
- анализ результатов

#### 2. **Предобработка данных (2.5 балла)**

**В рамках выполнения задания, вам необходимо:**
   - Провести токенизацию; по необходимости/желанию: удаление стоп-слов, приведение слов к нормальной форме (лемматизация/стемминг)
   - Подготовить данные для использования в моделях машинного обучения (например, преобразовать тексты в векторы с помощью `TF-IDF`, `CountVectorizer` или другого метода **(выбирая метод, обязательно опишите его)**).

#### 3. **Обучение моделей и анализ результатов (2.5 балла)**

**В рамках выполнения задания, вам необходимо:**
   - Построить и сравнить результаты моделей по метрике **accuracy**:
     - Логистическая регрессия
     - Любая другая модель машинного обучения — на ваш выбор
   - Для моделей подобрать гиперпараметры
   - Добиться улучшения качества относительно бейзлайна
   - Сделать прогноз на данных соревнования


In [25]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline, FeatureUnion

try:
    from google.colab import files
except ImportError:
    files = None
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

## Обработка данных

**Выбор подхода**
- новости - тексты однотипные по стилю, но отличающиеся как раз по лексике, так как в новостях много ключевых слов маркеров, поэтому эффективным будет подход TF-IDF, потому что текст по применении этого метода представляется в виде чисел, слова важные с лексической точки зрения получают большиий вес, а общие - меньший - проверяется насколько слово редкое в корпусе и насколько оно важно в документе
  - вес слова увеличивается, если оно часто встречается в документе (TF),
  - уменьшается, если слово встречается во многих документах корпуса (IDF)
- лемматизация может быть долгим процессом и не давать сильного прироста, на данном этапе не будем к ней прибегать
- так как TfidfVectorizer умеет не учитывать частые слова через max_df, то пока не будем вводить стоп-слова
- будем использовать линейные модели, так как они наделяют высоким весом слова-маркеры и отрицательный вес дают словам других тематик

In [4]:
df = pd.read_csv('train_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26998 entries, 0 to 26997
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   source    26998 non-null  object
 1   section   26998 non-null  object
 2   class_id  26998 non-null  int64 
 3   url       26998 non-null  object
 4   title     26998 non-null  object
 5   text      26998 non-null  object
 6   date      26998 non-null  object
 7   chars     26998 non-null  int64 
dtypes: int64(2), object(6)
memory usage: 1.6+ MB


- в модели будем использовать полный текст - заголовок и сам текст
- источник новостей не будем использовать, так как он сразу будет указывать на рубрику строительство
- возможна корреляция длины текста от тематики, но вряд ли вклад будет значительным
-остальные признаки также могут быть либо источником утечки или шума, не будем их использовать
- class_id - целевая переменная


In [6]:
#объединим текст и заголовок
def clean_text(text):
  text = str(text)
  text = text.lower()  # <-- lowercase=True (по сути)
  text = re.sub(r"\s+", " ", text).strip()
  return text

def build_full_text(df):
  return (df['title'] + '. ' + df['text']).apply(clean_text)

df['full'] = build_full_text(df)
df[['full', 'class_id']].head()

                                                full  class_id
0  минобороны сообщило об уничтожении украинского...         0
1  российские войска заняли населенный пункт в зо...         0
2  российская армия заняла населенный пункт в днр...         0
3  появились подробности о массированном и группо...         0
4  россиянин едва не лишился полового органа из-з...         0

In [9]:
#проверим, есть ли шаблонные фразы для предварительной очистки
def find_frequent_lines(texts, min_len=20, top_k=30, min_docs=300):
    """
    Ищем часто повторяющиеся строки (разделённые переводами строк).
    min_docs - сколько документов должно содержать строку, чтобы считать её мусором
    """
    line_doc_count = Counter()
    for t in texts:
        lines = set([ln.strip() for ln in str(t).split("\n") if len(ln.strip()) >= min_len])
        for ln in lines:
            line_doc_count[ln] += 1

    frequent = [(ln, c) for ln, c in line_doc_count.items() if c >= min_docs]
    frequent = sorted(frequent, key=lambda x: x[1], reverse=True)[:top_k]
    return frequent

freq_lines = find_frequent_lines(df['full'].tolist(), min_len=25, top_k=20, min_docs=300)
freq_lines

#lines_to_remove = [ln for ln, c in freq_lines]

#def remove_frequent_lines(text: str, bad_lines) -> str:
#    if not text:
#       return ""

#    parts = [p.strip() for p in text.split("\n")]
#    parts = [p for p in parts if p not in bad_lines]
#    return basic_clean(" ".join(parts))

#df["full_clean"] = df["full"].apply(lambda t: remove_frequent_lines(t, lines_to_remove))



[]

In [10]:
#отдельно для шапки РИА, так как там дата посреди шаблона
ria_header_re = re.compile(r"^[А-ЯЁA-Z\s\-]+,\s*\d{1,2}\s+[а-яё]+\s*[-–—]\s*риа\s+новости\.\s*", flags=re.IGNORECASE)

def remove_ria_header(text: str) -> str:
    if not text:
        return ""
    return clean_text(ria_header_re.sub("", text))

df['full_clean'] = df['full'].apply(remove_ria_header)


In [11]:
df.head()

  source section  class_id                                                url  \
0  lenta  Россия         0  https://lenta.ru/news/2026/01/30/minoborony-so...   
1  lenta  Россия         0  https://lenta.ru/news/2026/01/30/rossiyskie-vo...   
2  lenta  Россия         0  https://lenta.ru/news/2026/01/30/rossiyskaya-a...   
3  lenta  Россия         0  https://lenta.ru/news/2026/01/30/poyavilis-pod...   
4  lenta  Россия         0  https://lenta.ru/news/2026/01/30/rossiyanin-ed...   

                                               title  \
0  Минобороны сообщило об уничтожении украинского...   
1  Российские войска заняли населенный пункт в зо...   
2     Российская армия заняла населенный пункт в ДНР   
3  Появились подробности о массированном и группо...   
4  Россиянин едва не лишился полового органа из-з...   

                                                text        date  chars  \
0  Минобороны сообщило об уничтожении катера ВСУ....  2026-01-30    649   
1  Вооруженные силы (ВС) Р

In [13]:
X = df['full_clean']
y = df['class_id']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print('Распределение по тренирочным данным', np.bincount(y_train))
print('Распределение по валидационным данным', np.bincount(y_val))

Распределение по тренирочным данным [2404 2398 2379 2420 2389 2407 2398 2414 2389]
Распределение по валидационным данным [596 602 621 580 611 593 600 586 611]


## Модель

### Бейзлайн

- посмотрим на отдельные слова через униграммы для быстрого и простого начала, легко интерпретировать
- максимальное число итераций = 200 - для скорости и сходимости оптимизации на признаках tfidf

In [14]:
base = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=2,
        max_df=0.95,
        max_features=100000
    )),
    ('clf', LogisticRegression(
        max_iter=200
    ))
])

base.fit(X_train, y_train)
pred = base.predict(X_val)
acc = accuracy_score(y_val, pred)
print(acc)


0.9387037037037037


### word TF-IDF + LogReg

разбиение по словам и словосочетаниям - усложнение может дать улучшение в качестве, так как некоторые словосочетания дают лучшее представление о лексике тематики

In [15]:
pipe_word = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True)),
    ('clf', LogisticRegression(max_iter=300, n_jobs=-1))
])

param_grid_word = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__min_df': [2, 5],
    'tfidf__max_df': [0.9, 0.95],
    'tfidf__max_features': [100000],
    'clf__C': [1.0, 3.0],
}

gs_word = GridSearchCV(
    pipe_word,
    param_grid=param_grid_word,
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=1
)

gs_word.fit(X_train, y_train)
print('Лучшие параметры:', gs_word.best_params_)
pred = gs_word.best_estimator_.predict(X_val)
acc_word = accuracy_score(y_val, pred)
print('Метрика:', acc_word)

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Лучшие параметры: {'clf__C': 3.0, 'tfidf__max_df': 0.9, 'tfidf__max_features': 100000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Метрика: 0.9455555555555556


### char TF-IDF + LinearSVC

последовательность символов, которая помогает без лемматизации определять морфологию слов в русском - полезно для разных форм слова, но такую модель сложнее интерпретировать и в модели много признаков

In [17]:
pipe_char = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        analyzer='char_wb'
    )),
    ('clf', LinearSVC())
])

param_grid_char = {
    'tfidf__ngram_range': [(3,5), (3,6)],
    'tfidf__min_df': [3],
    'tfidf__max_df': [0.95],
    'tfidf__max_features': [150000],
    'clf__C': [1.0]
}

gs_char = GridSearchCV(
    pipe_char,
    param_grid=param_grid_char,
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=2
)

gs_char.fit(X_train, y_train)
print('Лучшие параметры:', gs_char.best_params_)
pred = gs_char.best_estimator_.predict(X_val)
acc_char = accuracy_score(y_val, pred)
print('Метрика:', acc_char)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
Лучшие параметры: {'clf__C': 1.0, 'tfidf__max_df': 0.95, 'tfidf__max_features': 150000, 'tfidf__min_df': 3, 'tfidf__ngram_range': (3, 6)}
Метрика: 0.9485185185185185


### Combo word+char FeatureUnion + LinearSVC

комбинирование предыдущих двух способов, должно дать выигрыш

In [24]:
word_vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2, max_df=0.95, max_features=100000)
char_vec = TfidfVectorizer(lowercase=True, analyzer='char_wb', ngram_range=(3,5), min_df=2, max_df=0.95, max_features=200000)

pipe_combo = Pipeline([
    ('features', FeatureUnion([
        ('word', word_vec),
        ('char', char_vec),
    ])),
    ('clf', LinearSVC())
])

param_grid_combo = {
    'clf__C': [1.0, 2.0],
    'features__word__ngram_range': [(1,1), (1,2)],
    'features__word__min_df': [2, 5],
}

gs_combo = GridSearchCV(
    pipe_combo,
    param_grid=param_grid_combo,
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=1
)

gs_combo.fit(X_train, y_train)
print('Лучшие параметры:', gs_combo.best_params_)
pred = gs_combo.best_estimator_.predict(X_val)
print('Метрика:', accuracy_score(y_val, pred))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Лучшие параметры: {'clf__C': 1.0, 'features__word__min_df': 2, 'features__word__ngram_range': (1, 2)}
Метрика: 0.9501851851851851


## Анализ ошибок

In [26]:
ID2LABEL = {
    0: "Россия",
    1: "Экономика",
    2: "Силовые структуры",
    3: "Бывший СССР",
    4: "Спорт",
    5: "Забота о себе",
    6: "Строительство",
    7: "Путешествия",
    8: "Наука и техника",
}

In [27]:
best_model = gs_combo.best_estimator_
y_pred = best_model.predict(X_val)
labels = sorted(ID2LABEL.keys())

cm = confusion_matrix(y_val, y_pred, labels=labels)
cm_df = pd.DataFrame(cm,
    index=[f'{i}:{ID2LABEL[i]}' for i in labels],
    columns=[f'{i}:{ID2LABEL[i]}' for i in labels]
)

cm_df

                     0:Россия  1:Экономика  2:Силовые структуры  \
0:Россия                  496           32                   21   
1:Экономика                27          543                   11   
2:Силовые структуры        13            7                  593   
3:Бывший СССР              22            1                    2   
4:Спорт                     0            1                    0   
5:Забота о себе             4            0                    1   
6:Строительство             0            0                    3   
7:Путешествия               5            3                    1   
8:Наука и техника           1            7                    2   

                     3:Бывший СССР  4:Спорт  5:Забота о себе  6:Строительство  \
0:Россия                        25        0                3                3   
1:Экономика                      3        4                3                2   
2:Силовые структуры              2        2                0                1   
3:Быв

In [29]:
classification_report(y_val, y_pred, target_names=[ID2LABEL[i] for i in labels], digits=4)

'                   precision    recall  f1-score   support\n\n           Россия     0.8732    0.8322    0.8522       596\n        Экономика     0.9141    0.9020    0.9080       602\nСиловые структуры     0.9353    0.9549    0.9450       621\n      Бывший СССР     0.9319    0.9431    0.9374       580\n            Спорт     0.9870    0.9918    0.9894       611\n    Забота о себе     0.9783    0.9899    0.9841       593\n    Строительство     0.9900    0.9933    0.9917       600\n      Путешествия     0.9713    0.9829    0.9771       586\n  Наука и техника     0.9655    0.9607    0.9631       611\n\n         accuracy                         0.9502      5400\n        macro avg     0.9496    0.9501    0.9498      5400\n     weighted avg     0.9497    0.9502    0.9499      5400\n'

In [30]:
pairs = []
for i, true_id in enumerate(labels):
    for j, pred_id in enumerate(labels):
        if true_id != pred_id and cm[i, j] > 0:
            pairs.append((cm[i, j], true_id, pred_id))
pairs.sort(reverse=True)

print('Какие пары больше всего путают? (истина - предсказание):')
for cnt, t, p in pairs[:10]:
    print(f'{cnt:>5} : {t} ({ID2LABEL[t]})  ->  {p} ({ID2LABEL[p]})')

Какие пары больше всего путают? (истина - предсказание):
   32 : 0 (Россия)  ->  1 (Экономика)
   27 : 1 (Экономика)  ->  0 (Россия)
   25 : 0 (Россия)  ->  3 (Бывший СССР)
   22 : 3 (Бывший СССР)  ->  0 (Россия)
   21 : 0 (Россия)  ->  2 (Силовые структуры)
   13 : 2 (Силовые структуры)  ->  0 (Россия)
   11 : 1 (Экономика)  ->  2 (Силовые структуры)
    9 : 0 (Россия)  ->  8 (Наука и техника)
    7 : 8 (Наука и техника)  ->  3 (Бывший СССР)
    7 : 8 (Наука и техника)  ->  1 (Экономика)


## Тестирование

In [20]:
import csv, sys
csv.field_size_limit(sys.maxsize)

test = pd.read_csv('test_news.csv', engine="python")
print(test.shape)

(26275, 1)


In [21]:
if {"title", "text"}.issubset(test.columns):
    test = build_full_text(test).apply(remove_ria_header)
elif "text" in test.columns:
    test = test["text"].apply(clean_text).apply(remove_ria_header)
elif test.shape[1] == 1:
    test = test.iloc[:, 0].apply(clean_text).apply(remove_ria_header)
else:
    raise ValueError("Ожидаются title/text, text или один текстовый столбец")

In [22]:
best_model = gs_combo.best_estimator_ #изменила, так как послений пайплайн
                                      #показал результат лучше
test_pred = best_model.predict(test)

In [23]:
submission = pd.DataFrame({
    "topic": test_pred,
    "index": range(len(test_pred))
})

submission.to_csv("submission.csv", index=False, encoding="utf-8-sig")
if files is not None:
    files.download("submission.csv")
else:
    print("Сохранено: submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Вывод

- был дан хороший по качеству страрт с униграммами - слова важны для определения тематики, так как лексика являются определяющей для выделения рубрики
- качество было улучшено при применении биграмм, получилось улучшить определение текста благодаря словосочетаниям, которые могут быть устойчивыми или часто встречающимися на новостных тематиках
- качество при использовании последовательности символов была ненамного улучшено - значит влияение оказывают различные формы слов
- наилучшее качество дала итоговая версия - скомбинированный вариант, так как она объединяет в себе 2 типа оценки: по словам и по формам слов

При этом:
- различие между модеями не такое большое, поэтому лучше искать варианты дальнейшего улучшения качества
- чаще всего модели ошибаются на теме экономики, России, Бывшего СССР, Силовых структур, наиболее точно - наука, спорт, путешествия, забота о себе, строительство, так как более уникальная лексика
- россия - самый широкий и размытый класс: туда попадает соц повестка, госуправление, региональные новости, связанные с экономикой и др.
- у бывшего ссср сильная привязка к России
- основная зона роста качества - улучшить различение Россия и темы: экономика, силовые структуры, Бывший СССР